In [1]:
import pandas as pd
from constants import DATA_PATH, EOS_FILE, SENTINEL_FILE

sentinel = pd.read_csv(DATA_PATH / SENTINEL_FILE)
eos = pd.read_csv(DATA_PATH / EOS_FILE)

In [2]:
sentinel = sentinel[sentinel['SM1 (%)'] != 50]
eos = eos[eos['SM1 (%)'] != 50]

In [3]:
from constants import X_cols_eos, X_cols_sentinel, y_col

X_sentinel = sentinel[X_cols_sentinel].values
X_eos = eos[X_cols_eos].values

y_sentinel = sentinel[y_col].values
y_eos = eos[y_col].values

In [4]:
import tensorflow as tf

I0000 00:00:1778441731.988395 1203472 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778441732.017495 1203472 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1778441732.733691 1203472 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [5]:
n_features = X_eos.shape[1]

models = {
#     "16, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(16, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
#     "8, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(8, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
#     "2, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(2, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
#     "4, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(4, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
    "16, Dropout, 8, Dropout": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(n_features, )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.09),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dropout(0.09),
    tf.keras.layers.Dense(1)
]),
    "16, Dropout": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(n_features, )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)
])
}

I0000 00:00:1778441733.580743 1203472 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6157 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [6]:
from model_experiments import PredictionIntervalEstimation

tf.keras.backend.clear_session()

eos_results = {}

for param_string, model in models.items():
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

    exp = PredictionIntervalEstimation(X_eos, y_eos, satellite="EOS-04")
    results = exp.run_experiment(model, model_param_string=param_string, optimizer=optimizer, epochs=1000)
    eos_results[param_string] = results

Results → /home/lmaosid/Desktop/major/experiments/classification_new_data/output/pi_estimation_uncensored


Upper model:   0%|          | 0/1000 [00:00<?, ?epoch/s]

I0000 00:00:1778441734.707984 1203575 service.cc:153] XLA service 0x72041c032450 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778441734.707999 1203575 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 13.2.0; Runtime: 12.4.0; Toolkit: 12.5.0; DNN: 9.3.0)
I0000 00:00:1778441734.719302 1203575 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778441734.790331 1203575 cuda_dnn.cc:461] Loaded cuDNN version 90300
I0000 00:00:1778441734.817615 1203575 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1529__.8


I0000 00:00:1778441735.915941 1203575 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1778441736.018048 1203574 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1529__.8


Upper model:   0%|          | 0/1000 [00:03<?, ?epoch/s, loss=17.7138, val_loss=16.3131]

...

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


16, Dropout: {
    "val": {
        "PICP": 0.966292,
        "MPIW": 31.920994
    },
    "test": {
        "PICP": 0.944134,
        "MPIW": 33.015774
    }
}


In [7]:
from constants import OUTPUT_PATH
import json

with open(OUTPUT_PATH / "pi_estimation_uncensored" / "EOS-04_metrics.json", "w") as f:
    json.dump(eos_results, f, indent=4)

In [8]:
from model_experiments import PredictionIntervalEstimation
sentinel_results = {}

for param_string, model in models.items():
    tf.keras.backend.clear_session()
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

    exp = PredictionIntervalEstimation(X_sentinel, y_sentinel, satellite="Sentinel-1")
    results = exp.run_experiment(model, model_param_string=param_string, optimizer=optimizer, epochs=1000)
    sentinel_results[param_string] = results

Results → /home/lmaosid/Desktop/major/experiments/classification_new_data/output/pi_estimation_uncensored


Upper model:   0%|          | 0/1000 [00:00<?, ?epoch/s]

I0000 00:00:1778441784.409305 1203576 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_39182__.8


I0000 00:00:1778441785.103955 1203576 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_39182__.8


Upper model:   0%|          | 0/1000 [00:02<?, ?epoch/s, loss=19.7207, val_loss=19.4612]

...

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 


16, Dropout: {
    "val": {
        "PICP": 0.934307,
        "MPIW": 39.485336
    },
    "test": {
        "PICP": 0.970803,
        "MPIW": 40.226471
    }
}


In [9]:
from constants import OUTPUT_PATH
import json

with open(OUTPUT_PATH / "pi_estimation_uncensored" / "Sentinel-1_metrics.json", "w") as f:
    json.dump(sentinel_results, f, indent=4)